In [ ]:
%pip install cleanlab tqdm  numpy pandas torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 22.7 MB/s eta 0:00:00


In [ ]:
!pip install "transformers<=4.48.0" "tokenizers<=0.21.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.16.1
    Uninstalling huggingface_hub-1.16.1:
      Successfully uninstalled huggingface_hub-1.16.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import pandas as pd
import numpy as np
import cleanlab
import torch
from transformers import pipeline, XLMRobertaTokenizer, AutoModelForSequenceClassification

In [ ]:
# 1. KHỞI TẠO TOKENIZER, MODEL
model_path = '5CD-AI/Vietnamese-Sentiment-visobert'

tokenizer = XLMRobertaTokenizer.from_pretrained(model_path, legacy=False)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Tạo pipeline
classifier = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True, # Lấy đầy đủ 3 cột xác suất
    device=0,               # CUDA
    truncation=True,        # Truncate sequences longer than model's max input size
    padding='max_length',   # Pad shorter sequences to the max_length
    max_length=512
)


# 2. ĐỌC VÀ LÀM SẠCH DỮ LIỆU TỪ FILE CSV
# Thứ tự nhãn cố định của mô hình 5CD-AI/Vietnamese-Sentiment-visobert: 0 -> NEG, 1 -> POS, 2 -> NEU
label_list = ['NEG', 'POS', 'NEU']
label_map = {label: idx for idx, label in enumerate(label_list)}

# Đọc dữ liệu từ file CSV
df = pd.read_csv("/content/d_data.csv")

print(f"Số lượng dòng ban đầu trong file: {len(df)}")

# Loại bỏ hàng rỗng và ép kiểu văn bản thành String
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str)
df = df[df['text'].str.strip() != ""] # Loại bỏ chuỗi khoảng trắng

print(f"Số lượng dòng sau khi xử lý dữ liệu trống: {len(df)}")

# Ánh xạ nhãn sang ID
df['label_id'] = df['label'].map(label_map)

# Loại bỏ nhãn lạ không trong [POS, NEU, NEG] nếu có
df = df.dropna(subset=['label_id'])
df['label_id'] = df['label_id'].astype(int)

labels = df['label_id'].to_numpy()
sentences = df['text'].tolist()


# 3. DỰ ĐOÁN XÁC SUẤT (PRED_PROBS) TỐI ƯU BATCH SIZE
print("\nĐang chạy mô hình để thu thập mảng xác suất dự đoán...")
pred_probs_list = []

# Chạy dự đoán song song theo Batch
outputs = classifier(sentences, batch_size=32)

for out in outputs:
    # Tạo một dictionary tạm để lưu kết quả
    # Ví dụ: {'NEG': 0.9, 'POS': 0.05, 'NEU': 0.05}
    score_dict = {item['label']: item['score'] for item in out}

    # Ép buộc trích điểm số theo thứ tự cố định: 0->NEG, 1->POS, 2->NEU
    ordered_scores = [score_dict['NEG'], score_dict['POS'], score_dict['NEU']]
    pred_probs_list.append(ordered_scores)

pred_probs = np.array(pred_probs_list)

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/471k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/390M [00:00<?, ?B/s]

Device set to use cuda:0


Số lượng dòng ban đầu trong file: 31007
Số lượng dòng sau khi xử lý dữ liệu trống: 31007

Đang chạy mô hình để thu thập mảng xác suất dự đoán...


In [ ]:
# 4. QUÉT LỖI VÀ GỢI Ý NHÃN MỚI BẰNG CLEANLAB
print("\n--- Tiến hành tìm lỗi nhãn bằng Cleanlab ---")

# Quét tìm các index bị sai nhãn
is_issue = cleanlab.filter.find_label_issues(
    labels=labels,
    pred_probs=pred_probs
)
issue_indices = np.where(is_issue)[0]

# Gợi ý nhãn chính xác bằng hàm argmax dựa trên xác suất cao nhất của mô hình
suggested_label_ids = np.argmax(pred_probs, axis=1)
df['suggested_label'] = [label_list[idx] for idx in suggested_label_ids]


--- Tiến hành tìm lỗi nhãn bằng Cleanlab ---


In [ ]:
# 5. IN KẾT QUẢ VÀ XUẤT FILE SẠCH (LOẠI BỎ DÒNG LỖI)
if len(issue_indices) > 0:
    print(f"\n Phát hiện {len(issue_indices)} mẫu dữ liệu bị gán sai nhãn gốc:")
    error_df = df.iloc[issue_indices][['text', 'label', 'suggested_label']]
    print(error_df.head(20)) # In ra tối đa 20 dòng lỗi


    # Thực hiện DROP các dòng có issue bằng cách dùng toán tử phủ định ~
    df_cleaned = df[~is_issue].copy()
    df_cleaned = df_cleaned[['text', 'label']] # Giữ lại nhãn gốc ban đầu

    # Xuất ra file dữ liệu sạch hoàn chỉnh sau khi drop
    df_cleaned.to_csv('/content/dataset_cleaned_drop.csv', index=False)
    print(f" Đã loại bỏ {len(issue_indices)} dòng lỗi.")
    print(f" Số lượng dòng còn lại trong file sạch: {len(df_cleaned)}")
    print(" Đã cập nhật và xuất dữ liệu sạch ra file '/content/dataset_cleaned_drop.csv'")
else:
    print("\n Không phát hiện lỗi nhãn nào. Giữ nguyên tập dữ liệu.")


 Phát hiện 1290 mẫu dữ liệu bị gán sai nhãn gốc:
                                                    text label suggested_label
24799  hồi nãy check khóa học thì thấy vẫn hiển thị b...   NEU             POS
24851         tài khoản mình hiện có email trong lịch sử   NEU             NEG
24856          bên app báo app đã cập nhật phiên bản mới   NEU             POS
24861  mới mở app thấy tin nhắn vẫn hiển thị bình thường   NEU             POS
24865            t mới coi điện thoại được gửi qua email   NEU             NEG
24875      tài khoản mình hiện có máy tính trong lịch sử   NEU             NEG
24878                  bên app báo email có lịch bảo trì   NEU             NEG
24887          mới mở app thấy máy tính có thông báo mới   NEU             POS
24888     bên app báo tài khoản xuất hiện trên trang chủ   NEU             NEG
24890      mới mở app thấy phim vẫn hiển thị bình thường   NEU             NEG
24892  mới mở app thấy đơn hàng vẫn hiển thị bình thường   NEU             NEG
24